1. *Data Description*

Dataset Overview:

(1) The dataset consists of two related tables: a player dataset and a session dataset.

(2) The player dataset contains 196 unique players and 7 variables describing demographics and play history.

(3) The session dataset contains 1535 recorded play sessions and 5 variables detailing session start/end times


data collection method: the data is collected passively from a Minecraft server. `players.csv` contains registration and profile information provided by users upon signing up. `sessions.csv` is an automated log of player connection and disconnection times, generated by the server software.

dataset 1: player demographics and profile
| Variable     | Type        | Description                                       |
| ------------ | ----------- | ------------------------------------------------- |
| experience   | factor      | Self-reported experience level                    |
| subscribe    | logical     | Indicates whether player subscribed to newsletter |
| hashedEmail  | character   | Anonymous unique player ID                        |
| played_hours | double      | Total hours spent playing                         |
| name         | character   | Player display name                               |
| gender       | factor      | Reported gender                                   |
| Age          | integer     | Age in years                                      |

dataset 2: player session logs
| Variable Name           | Type                      | Meaning                                                                                                                                                             |
| ----------------------- | ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **hashedEmail**         | character                    | A hashed (anonymized) unique identifier for each player. This is used to link sessions to the player dataset.                                                       |
| **start_time**          | character         | The recorded start time of the session, formatted as `DD/MM/YYYY HH:MM`. Requires conversion to a proper datetime type for analysis.                                |
| **end_time**            | character         | The recorded end time of the session, also formatted as `DD/MM/YYYY HH:MM`. Paired with `start_time` to compute session duration.                                   |
| **original_start_time** | double | Start time recorded as a Unix-like timestamp in scientific notation (e.g., `1.71977E+12`). This provides a more precise, timezone-standardized way to measure time. |
| **original_end_time**   | double | End time recorded in the same epoch timestamp format as `original_start_time`. Useful for precise duration calculations and time-series modeling.                   |


In [2]:
library(tidyverse)
library(repr)
library(tidymodels)
library(GGally)
library(ISLR)
options(repr.matrix.max.rows = 6)
options(scipen = 999)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
── Attaching packages ────────────────────────────────────── tidymodels 1.1.1 ──

✔ broom        1.0.6     ✔ rsample      1.2.1
✔ dials        1.3.0     ✔ tune         1.1.2
✔ infer        1.0.7     ✔ workflows    1.1.4
✔ modeldata    1.4.0     ✔ workflowsets 1.0.1
✔ parsnip      1.2.1     ✔ yardstick    1.3.1
✔ recipes      1.1.0     

── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filt

*2. Questions*

Broad Question：

Question 3: We are interested in demand forecasting, namely, what time windows are most likely to have large number of simultaneous players. This is because we need to ensure that the number of licenses on hand is sufficiently large to accommodate all parallel players with high probability.


Specific Question：

“Can the total number of hours a player has accumulated (from players.csv) and the duration of their previous sessions (from sessions.csv) predict whether they will start a new session in the next 24 hours?”

Predictive method:

K-nearest neighbors classification (only one method)


How the data will help address the question：

To address this question, I will combine information from `players.csv` and `sessions.csv` using `hashedEmail`. From these data, I will create predictors such as a player’s total hours played and their average session duration. I will also compute a binary response variable that indicates whether a player begins another session within 24 hours of their most recent one. Using tidyverse tools like `mutate`, `summarize`, and `group_by`, I will construct a tidy dataset, which will then be used to train a K-nearest neighbors classification model to predict whether a player will return within 24 hours.